# Phase4: ルール補正・確信度・「要確認」 — AIは間違える前提でシステムを作る

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: AIの読み間違いに備え、書式ルールでの補正と、「自信がなければ人に確認」の仕組みを作る回です。
> - **たとえ話**: 自信がないときは先輩に聞く新人。聞く回数と、間違える回数のバランスを決める。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-5（確信度と要確認）・2-6（ルール補正）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 25分 ＋ 実行 約1分

## このプロジェクトのゴール（おさらい）
駐車場のナンバー読み取り精算システムのミニ版を作り、**AIはシステムの一部品**であることを確かめます。

| Phase | 分かったこと |
|---|---|
| 1 | AIなし: きれいな条件ではほぼ完璧、悪いと急に壊れる |
| 2 | 検出AI: 検出はほぼ100%。ただし読み取りには四隅が1〜2px以内の精度が必要 |
| 3 | 読み取りAI（CRNN）: Medium 0.34→0.90。**Hard は画像から情報が失われ、読めない。** 確信度は粗かった |

## このNotebookのゴール
**AIの読み取りが「間違える」ことを前提に、安全に使える形にします。** ここが、AIを**システムの部品**にする核心です。

1. **書式ルール**で、読み取り結果を**検査**する（ありえない文字列を弾く）。直せるなら**補正**する
2. **確信度**（どれくらい自信があるか）を作り、**しきい値**で3つに**振り分ける**
   - **自動処理**: 自信があり、書式も正しい → そのまま進める
   - **要確認**: 自信がない → 候補を添えて、人に確認してもらう
   - **読み取り不可**: 書式が直せない → 人が手で入力する
3. **「自動処理する割合」と「その中の誤り率」のトレードオフ**（カバレッジ曲線）を見て、しきい値を決める

> 💡 **設計の原則: 誤って課金するより、「要確認」に回すほうを優先する。**
> 誤課金（別の車として課金する）は、お客さまへの損害と信頼の低下になりますが、要確認は**係員の手間が増える**だけです。

## キーワード
| 用語 | 意味 |
|---|---|
| **書式検査** | 読み取り結果が、板の書式（地名の一覧、桁数、文字の種類）に合うかを調べること。機械学習ではなく、人が書いたルール |
| **補正** | 書式に合わない文字を、候補の中から書式に合うものに差し替えること |
| **確信度** | AIが「どれくらい自信があるか」を表す数。値が大きいほど自信がある |
| **しきい値** | 確信度が、これ以上なら「自動処理」とする境目の値 |
| **自動処理率（カバレッジ）** | 自動処理に回った割合 |
| **自動処理の誤り率（リスク）** | 自動処理に回ったもののうち、読み間違いだった割合。**最も重要** |
| **カバレッジ曲線** | しきい値を動かしたときの、「自動処理率」と「自動処理の誤り率」の関係を描いた曲線 |
| **AUROC** | 確信度が、正解と誤りをどれくらい見分けられるかの指標（0.5=偶然、1=完全） |
| **較正（キャリブレーション）セット** | しきい値を決めるための、正解付きのデータ。評価用とは別に用意する |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch, PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from ai_detect.data import to_tensor, target_to_corners
from ai_detect.model import make_model as make_detector
from ai_ocr.data import EvalItems
from ai_ocr.infer import load_plates, read_plates
from ai_ocr.model import CRNN
from rules.format import violations, is_valid, split
from rules.correct import correct
from rules.confidence import confidences
from rules.decision import (AUTO, REVIEW, REJECT, apply, auroc, choose_threshold, choose_threshold_by_cost, decide, risk_coverage)

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 評価結果（src/eval_phase4.py の出力）を読む。ev=評価データ(valの101枚目以降)、cal=較正セット(しきい値決定用)
RULES = ROOT / "outputs" / "rules"
if not (RULES / "seed0_samples.csv").exists():
    raise SystemExit("評価結果がありません。ターミナルで  .venv/bin/python src/make_eval_sets.py --val 0 --test 0 --cal 1000  と  .venv/bin/python src/eval_phase4.py  を実行してください。")
ev = pd.concat([pd.read_csv(RULES / f"seed{s}_samples.csv") for s in range(3)]); ev = ev[ev.idx >= 100].reset_index(drop=True)
cal = pd.concat([pd.read_csv(RULES / f"seed{s}_cal_samples.csv") for s in range(3)]).reset_index(drop=True)
MIX = {"均等(1:1:1)": {"easy": 1/3, "medium": 1/3, "hard": 1/3}, "昼間中心(0.6:0.3:0.1)": {"easy": 0.6, "medium": 0.3, "hard": 0.1}}
def weights(df, mix):
    n = df.difficulty.value_counts()
    return df.difficulty.map(lambda d: MIX[mix][d] / n[d]).to_numpy()
print(f"評価データ {len(ev)}枚（3シード分）／較正セット {len(cal)}枚（3シード分）")

上のセルは、道具の読み込みと、これまでの評価結果の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

> 📌 前提: Phase2（検出器）と Phase3（OCR）の学習済みモデルがあり、`src/eval_phase4.py` を実行して評価結果（`outputs/rules/`）ができていること。
> 「評価データ」は val の101枚目以降（各400枚×3シード）、「較正セット」は val のシードの500番目以降から作った**しきい値決定専用**のデータ（各1,000枚×3シード）です。**test は使いません。**

## 1. 書式ルール — 「ありえない読み取り」を弾く

ナンバーには**書式**があります（このプロジェクトでは、簡略化した書式）。

| 部分 | ルール |
|---|---|
| 地名（2文字） | 沖縄・那覇・品川・横浜・大阪のどれか |
| 分類番号（3文字） | 1文字目は1〜9、2文字目は数字、3文字目は数字または英字 |
| かな（1文字） | 決まった10文字のどれか |
| 一連番号 | `12-34`（数字4桁の中央にハイフン）か、`・・12` のように上位を「・」で埋めた4文字 |

次のセルで、いくつかの文字列を検査します。**違反の理由**が出ます。

In [ ]:
tests = ["沖縄300あ12-34", "沖縄300あ・・12", "沖縄3O0あ12-34", "沖縄300あ1234", "東京300あ12-34", "沖縄300ん12-34", "沖縄300あ12"]
rows = [{"文字列": t, "書式": "✔ 正しい" if is_valid(t) else "✘ 違反", "理由": " / ".join(violations(t))} for t in tests]
pd.DataFrame(rows)

**見るポイント**
- 「3**O**0」（英字のオー）、「1234」（ハイフンなし）、存在しない地名やかな、文字数の違い — **どれも弾けます**。
- 弾く理由が**文章で出る**ので、人が確認するときの手がかりにもなります。
- ⚠ ただし、「3**8**0」と「3**6**0」のように、**どちらも書式に合う間違い**は、ルールでは見つけられません（あとで効いてきます）。

## 2. 検査の効果 — 弾いたものは、本当に間違いだったか

AIの読み取り結果を、評価データ全部（3シード×1,200枚）に検査をかけます。**書式に合わなかったもの**は、実際にどれくらい間違いだったでしょうか。

| 難易度 | そのまま正しい書式 | 補正して正しい書式 | 直せない | 完全一致（補正前） | 完全一致（補正後） | 補正した画像の正解率 |
|---|---|---|---|---|---|---|
| easy | 0.987 | 0.000 | 0.013 | 0.977 | 0.977 | nan（0枚）|
| medium | 0.969 | 0.001 | 0.030 | 0.897 | 0.897 | 0.000（1枚）|
| hard | 0.706 | 0.031 | 0.263 | 0.081 | 0.082 | 0.027（37枚）|

次のセルで、「直せない（読み取り不可）」と判定した画像のうち、**実は正解だった数**を数えます。

In [ ]:
inv = ev[~ev.valid_final]
print(f"書式が直せず『読み取り不可』にした画像: {len(inv)}枚（評価データ全体の {len(inv) / len(ev):.1%}）")
print(f"そのうち、AIの読み取りが実は正解だった画像: {int(inv.exact_raw.sum())}枚")
print("難易度別の内訳:", inv.difficulty.value_counts().to_dict())

**重要な発見**
- 書式に合わないものは、**100%間違い**でした（正解の文字列は必ず書式に合うので、当然です）。つまり、**書式検査は「安全な弾き方」**です。**正しい読み取りを誤って弾くことが、ありません**。
- 弾いた画像の大部分は **Hard**（文字数が違う、など）。これを「**読み取り不可**」として人に回すだけで、**自動処理に混ざる誤りが大きく減ります**。
- 検査は、**AIの出力を「信用しない」ための、安いセーフティネット**です。

## 3. 補正 — 直せるものは直す（けれど…）

書式に合わなかったとき、**候補の中から、書式に合う文字に差し替えられないか**を試します。
CRNNは、各文字を出力した位置で、**2位・3位…の候補**も持っています（確率つき）。たとえば「3**O**0」の「O」の位置で、2位が「0」なら、そちらに差し替えられます。

実際に、補正が働いた画像の1つで、その様子を見てみましょう。

In [ ]:
cor = ev[(ev.seed == 0) & (ev.status == "corrected")]
r0 = cor.iloc[0]
items = EvalItems("val", 0, 500).items
k = next(i for i, (d, p, lab) in enumerate(items) if d == r0.difficulty and lab["image"] == r0.image)
d_, path_, lab_ = items[k]
det = make_detector("resnet18").to(device); det.load_state_dict(torch.load(ROOT / "outputs" / "detect" / "resnet18_all_s0_best.pt", map_location=device)); det.eval()
with torch.no_grad():
    corners = target_to_corners(det(to_tensor(Image.open(path_))[None].to(device)).cpu().numpy()[0])
crnn = CRNN().to(device); crnn.load_state_dict(torch.load(ROOT / "outputs" / "ocr" / "crnn_all_cs3_s0_best.pt", map_location=device))
reading = read_plates(crnn, device, load_plates([items[k]], [corners]))[0]
res = correct(reading.emissions, reading.text)

print("正解 :", lab_["text"].replace(" ", ""))
print("AIの読み取り :", reading.text, "→ 書式違反:", violations(reading.text))
print("補正後 :", res.text, f"（{res.status}）", "／ 差し替え:", [f"{i + 1}文字目 {a}→{b}" for i, a, b in res.changes])
rows = [{"文字目": i + 1, "読んだ文字": e.char, "確率": round(e.prob, 2), "次点の候補": ", ".join(f"{c}({p:.2f})" for c, p in e.alts[1:4])} for i, e in enumerate(reading.emissions)]
display(pd.DataFrame(rows).set_index("文字目"))
Image.open(path_).convert("RGB")

**見るポイント**
- 「次点の候補」の列に、2位以下の文字と確率があります。地名の1文字目など、**書式に合わない文字**が、候補の中の**書式に合う文字**に差し替えられています。
- ところが、**差し替えた結果は、正解と違います**。この画像は**Hard**で、文字そのものが壊れて読めないため、書式に合うように**でっち上げただけ**です。

### 補正は、本当に役に立ったのか
評価データ全体で、補正が働いた画像を数えます。

In [ ]:
c = ev[ev.status == "corrected"]
print(f"補正が働いた画像: {len(c)}枚（評価データ全体の {len(c) / len(ev):.2%}）。うち、補正後に正解になったのは {int(c.exact_final.sum())}枚")
print("難易度別:", c.difficulty.value_counts().to_dict())
print(f"完全一致: 補正前 {ev.exact_raw.mean():.4f} → 補正後 {ev.exact_final.mean():.4f}")

**重要な発見（意外かもしれません）**
- **補正が働いたのは、ごくわずか**（全体の約1%）で、**ほぼ全部が Hard。補正後に正解になったのは、ほとんどありません。**
- 理由: 書式に合わない読み取りは、**文字が壊れている**場合がほとんど。壊れた文字を書式に合うように差し替えても、**「もっともらしい間違い」**ができるだけです。
- 危険な点: 補正すると**書式に合う**ので、**検査を通り抜けて**しまいます。**「もっともらしい嘘」は、人間にも見抜きにくい**。
- → **補正は、期待したほど役に立たない。** 書式を使うなら、**「検査して弾く」ことの価値が大きく、「補正して直す」価値は小さい**。補正した結果は、**自動処理にしない**（要確認に回す）のが安全です。

> 💡 一般に、**自動で「直す」処理は、元が壊れているときに、もっともらしい嘘を作る**危険があります。直したものは、必ず人の目に触れる設計にします。

## 4. 確信度 — 「どれくらい自信があるか」を作る

自動処理するかどうかは、**確信度**で決めます。Phase3 の「文字ごとの最小確率」は粗かったので、**4通り**の作り方を比べます。

| 名前 | 作り方 | ねらい |
|---|---|---|
| **min_max** | 各文字の確率（連続する位置のうち最大）の**最小値** | Phase3の方法。1文字でも自信がなければ低い |
| **min_sum** | 各文字の確率（連続する位置の**合計**）の最小値 | 確率が隣の位置に分かれても拾う |
| **margin** | 1位と2位の確率の**差**の最小値 | 「迷っている」文字を見つける |
| **seq** | **文字列全体の確率**（CTCで、その文字列になるすべての並べ方の確率の合計） | 文字列として、全体で自信があるか |

**AUROC**で、各確信度が「**正解と誤りを見分ける力**」を比べます（1に近いほど良い。0.5は偶然）。較正セット（3,000枚×3シード）で測りました。

| 確信度 | easy | medium | hard | 全体 |
|---|---|---|---|---|
| min_max | 0.952 | 0.876 | 0.901 | 0.976 |
| min_sum | 0.952 | 0.873 | 0.902 | 0.976 |
| margin | 0.956 | 0.864 | 0.876 | 0.967 |
| seq | 0.951 | 0.868 | 0.910 | 0.981 |

**見るポイント**
- **4つの確信度に、大きな差はありません**（難易度ごとに0.86〜0.96）。**seq（文字列全体の確率）は、Hard と全体で最良**ですが、Medium では min_max がわずかに上（0.876 vs 0.868）。**margin（1位と2位の差）**は、Easy では最良なのに、Medium・Hard・全体では最も弱い。
- 「全体」の数字が高い（0.98）のは、**難易度が混ざっている**ため（Hard は低く、Easy は高い確信度に偏るから）。**難易度ごとの値（0.87〜0.95）のほうが、実態に近い**。
- → 確信度は**手がかり**にはなるが、**完璧ではない**。間違いの一部は、自信たっぷりに出てくる（あとの「残る誤り」）。

次のグラフで、**正解した画像**と**誤った画像**の確信度（seq）の分布を比べます（較正セット、書式が正しいものだけ）。

In [ ]:
x = cal[cal.valid_final]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=False)
for ax, d in zip(axes, ["easy", "medium", "hard"]):
    s = x[x.difficulty == d]
    bins = np.linspace(0, 1, 21)
    ax.hist(s[s.exact_final].fin_seq, bins=bins, alpha=0.65, label="正解", color="C2")
    ax.hist(s[~s.exact_final].fin_seq, bins=bins, alpha=0.75, label="誤り", color="C3")
    ax.set(title=f"{d}: 確信度(seq)の分布", xlabel="確信度（seq）", yscale="log"); ax.legend(fontsize=8)
fig.tight_layout();

**見るポイント**
- **誤り（赤）は、確信度が低い側に多い**が、**高い側にも少し混ざっています**（縦軸は対数なので、少数でも見えます）。
- 正解（緑）と誤り（赤）は、**完全には分かれません**。分かれていない部分に、しきい値を置くと、**「自動処理の誤り」か「要確認が増える」か**のどちらかを選ぶことになります。**これがトレードオフ**です。

## 5. 振り分け — 自動処理／要確認／読み取り不可

3つの振り分けは、次のルールで決めます。

```
書式が直せない               → 読み取り不可（人が手で入力）
書式OK かつ 確信度 ≥ しきい値 → 自動処理
書式OK かつ 確信度 < しきい値 → 要確認（候補を添えて人へ）
```

### カバレッジ曲線を読む
**しきい値を動かす**と、「自動処理する割合（横軸）」と「自動処理の中の誤り率（縦軸）」が変わります。
- しきい値を**厳しく**（高く）→ 自動処理は**減る**が、誤りも**減る**（曲線の左側）
- しきい値を**緩く**（低く）→ 自動処理は**増える**が、誤りも**増える**（曲線の右側）
**左下に近い曲線ほど、良い方式**です（少ない誤りで、多く自動処理できる）。

次のグラフは、3つの方式を比べます。
- **A**: Phase3のまま（書式検査なし、確信度=最小確率）
- **B2**: 書式検査あり（補正しない）、確信度=seq
- **C**: 書式検査＋補正、確信度=seq

左は「均等（Easy:Medium:Hard = 1:1:1）」、右は「**昼間中心**（0.6:0.3:0.1）」という**実運用に近い**難易度の混ざり方です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
e0 = ev[ev.seed == 0]
for ax, mix in zip(axes, MIX):
    w = weights(e0, mix)
    for name, ok, valid, conf in [("A: Phase3のまま（検査なし・最小確率）", "exact_raw", None, "raw_min_max"),
                                  ("B2: 書式検査のみ・seq", "exact_raw", "raw_valid", "raw_seq"),
                                  ("C: 書式検査＋補正・seq", "exact_final", "valid_final", "fin_seq")]:
        v = e0[valid].to_numpy() if valid else None
        cov, err, _ = risk_coverage(e0[conf], e0[ok], v, w)
        ax.plot(cov, err, label=name, lw=1.8)
    ax.axhline(0.01, color="gray", ls=":", lw=1); ax.text(0.01, 0.012, "目標の誤り率 1%", fontsize=8, color="gray")
    ax.set(title=f"カバレッジ曲線: {mix}", xlabel="自動処理率（カバレッジ）", ylabel="自動処理した中の誤り率", ylim=(0, 0.2), xlim=(0, 1)); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8); fig.tight_layout();

**見るポイント**
- **A（Phase3のまま）**は、どれだけ絞っても**誤り率が1%より下がりません**（曲線が下がりきらない）。**確信度の高いところにも、誤りが混ざっている**ため。
- **B2・C は、Aより明らかに下**（良い）。**書式検査と、seq確信度**で、同じ誤り率でも**ずっと多く自動処理できる**。
- **B2とCは、ほぼ重なっています。** → **補正は、効いていません**（先ほどの発見のとおり）。
- 右（昼間中心）のほうが、全体に良い（Easy・Mediumが多いから）。**運用の条件（難易度の混ざり方）で、結果が大きく変わる**ことに注意。

## 6. しきい値の決め方 — 誤り率の目標を決める

「自動処理の誤り率を、**1%以下**にしたい」という目標があるとします。**目標を満たす中で、自動処理がいちばん多くなるしきい値**を選びます。

**⚠ しきい値は、評価データとは別のデータで決める**（評価データで決めると、評価が甘くなる＝カンニング）。そのデータは**何枚必要でしょうか？** 2通りで比べました。
- **S**: 検証セットの**先頭100枚/難易度（300枚）**で決める
- **C**: **較正セット（3,000枚）**で決める
どちらも、**101枚目以降の評価データ**で、実際の結果を測ります（3シード平均、昼間中心）。目標は 0.5%・1%・2%。

**S（300枚で決める）**

| 方式 | 目標の誤り率 | 自動処理率 | 実現した誤り率 | 要確認率 | 読み取り不可率 | 全体に対する誤自動処理率 |
|---|---|---|---|---|---|---|
| A: 補正なし・検査なし・最小確率（Phase3） | 0.020 | 0.601 | 0.0316 | 0.399 | 0.000 | 0.0179 |
| A: 補正なし・検査なし・最小確率（Phase3） | 0.010 | 0.318 | 0.0294 | 0.682 | 0.000 | 0.0085 |
| A: 補正なし・検査なし・最小確率（Phase3） | 0.005 | 0.158 | 0.0216 | 0.843 | 0.000 | 0.0018 |
| B2: 書式検査のみ（補正しない）・seq | 0.020 | 0.867 | 0.0234 | 0.086 | 0.047 | 0.0204 |
| B2: 書式検査のみ（補正しない）・seq | 0.010 | 0.859 | 0.0195 | 0.094 | 0.047 | 0.0168 |
| B2: 書式検査のみ（補正しない）・seq | 0.005 | 0.808 | 0.0147 | 0.145 | 0.047 | 0.0122 |
| C: 補正＋検査・seq | 0.020 | 0.868 | 0.0234 | 0.089 | 0.043 | 0.0204 |
| C: 補正＋検査・seq | 0.010 | 0.859 | 0.0195 | 0.098 | 0.043 | 0.0168 |
| C: 補正＋検査・seq | 0.005 | 0.808 | 0.0147 | 0.148 | 0.043 | 0.0122 |

**C（3,000枚で決める）**

| 方式 | 目標の誤り率 | 自動処理率 | 実現した誤り率 | 要確認率 | 読み取り不可率 | 全体に対する誤自動処理率 |
|---|---|---|---|---|---|---|
| A: 補正なし・検査なし・最小確率（Phase3） | 0.020 | 0.390 | 0.0365 | 0.610 | 0.000 | 0.0064 |
| A: 補正なし・検査なし・最小確率（Phase3） | 0.010 | 0.113 | 0.0274 | 0.887 | 0.000 | 0.0008 |
| A: 補正なし・検査なし・最小確率（Phase3） | 0.005 | 0.106 | 0.0275 | 0.894 | 0.000 | 0.0008 |
| B2: 書式検査のみ（補正しない）・seq | 0.020 | 0.843 | 0.0151 | 0.111 | 0.047 | 0.0129 |
| B2: 書式検査のみ（補正しない）・seq | 0.010 | 0.793 | 0.0083 | 0.160 | 0.047 | 0.0066 |
| B2: 書式検査のみ（補正しない）・seq | 0.005 | 0.721 | 0.0046 | 0.232 | 0.047 | 0.0033 |
| C: 補正＋検査・seq | 0.020 | 0.843 | 0.0151 | 0.114 | 0.043 | 0.0129 |
| C: 補正＋検査・seq | 0.010 | 0.793 | 0.0083 | 0.163 | 0.043 | 0.0066 |
| C: 補正＋検査・seq | 0.005 | 0.721 | 0.0046 | 0.235 | 0.043 | 0.0033 |

**見るポイント（とても大事）**
- **S（300枚）**で決めたしきい値は、**実際の誤り率が目標を大きく超えます**（たとえば、C-seq で目標0.5% → **実際1.5%**）。選択データが少ないと、**誤り率の見積もり自体がぶれる**ため、「目標を満たす」と見えたしきい値が、**運が良かっただけ**のことがあるからです。
- **C（3,000枚）**で決めると、**実際の誤り率がほぼ目標どおり**（目標0.5% → 実際0.46%、1% → 0.83%）。
- **教訓**: **しきい値の決定には、十分な数の正解付きデータが必要**。誤りの少ない領域（誤り率0.5%を確かめるには、数千件が要る）ほど、多く必要になる。
  - 今回は**合成データ**なので、**較正セットを好きなだけ作れる**。実際のシステムでは、正解付きデータは**高価**。少ないデータでしきい値を決めるなら、**誤り率の「上限」**（信頼区間）で判断する、**目標より厳しめに決める**といった**安全側のマージン**が必要になります。

## 7. 「コスト」で決める — 誤課金を重く見る

「誤り率1%以下」という目標は、どこから来るのでしょう。**本当に知りたいのは、損失の大きさ**です。

- **要確認**: 係員が確認する手間（1件=1とする）
- **誤課金**: 自動処理で読み間違えた損害（1件= **N** とする）。お客さまへの損害・信頼・対応コストを含む

**1台あたりの期待コスト = 人が対応する割合×1 ＋ 自動処理して間違えた割合×N** を最小にするしきい値を選びます。N（誤課金の重さ）が**大きいほど**、**自動処理を絞る**はずです。確かめましょう（昼間中心、C: 書式検査＋補正・seq）。

In [ ]:
rows = []
w_cal, w_ev = weights(cal, "昼間中心(0.6:0.3:0.1)"), weights(ev, "昼間中心(0.6:0.3:0.1)")
for N in [1, 5, 20, 100, 500]:                              # ← 演習で変えてみる（誤課金1件が、要確認1件の何倍の損害か）
    tau = choose_threshold_by_cost(cal.fin_seq, cal.exact_final, cal.valid_final, 1.0, N, w_cal)
    r = apply(ev.fin_seq, ev.exact_final, ev.valid_final, tau, w_ev)
    cost = (1 - r["自動処理率"]) + r["誤自動処理率"] * N
    rows.append({"誤課金の重さ N": N, "しきい値": round(tau, 3), "自動処理率": round(r["自動処理率"], 3), "自動処理の誤り率": round(r["自動処理の誤り率"], 4),
                 "要確認・読み取り不可率": round(1 - r["自動処理率"], 3), "1台あたりの期待コスト": round(cost, 3)})
cost_df = pd.DataFrame(rows).set_index("誤課金の重さ N")
cost_df

**見るポイント**
- **N が大きくなるほど**、しきい値が**上がり**、**自動処理率が下がり**、**自動処理の誤り率も下がります**。**誤課金を重く見るほど、人に回す**。これが、設計の原則「**誤課金より要確認**」の、数字での表れです。
- N=1（誤りも要確認も同じ重さ）なら、ほぼ**全部自動処理**したほうが得、N=500 なら、**かなり慎重に**。**N の値は、運用する人が決める**（技術では決まらない）。
- 「誤り率1%以下」のような目標は、実は**N（損害の重さ）の裏返し**です。

**方式ごとの比較**（N=20、昼間中心、較正セットで決定、101枚目以降で評価）

| 方式 | しきい値 | 自動処理率 | 自動処理の誤り率 | 1台あたりの期待コスト |
|---|---|---|---|---|
| A: 補正なし・検査なし・最小確率（Phase3） | 0.6767 | 0.755 | 0.0200 | 0.5434 |
| B2: 書式検査のみ（補正しない）・seq | 0.5428 | 0.758 | 0.0065 | 0.3424 |
| C: 補正＋検査・seq | 0.5428 | 0.758 | 0.0065 | 0.3424 |

- **C（書式検査＋補正・seq）が最も期待コストが小さく**、A（Phase3のまま）が最大です。**書式検査とseq確信度で、コストが下がった**。

## 8. 残る誤り — ルールでも確信度でも防げないもの

最後に、**自動処理に回ったのに、間違えたもの**（「自信満々の誤り」）を調べます。昼間中心・目標1%の設定です。

In [ ]:
w_c = weights(cal, "昼間中心(0.6:0.3:0.1)")
tau = choose_threshold(cal.fin_seq, cal.exact_final, cal.valid_final, 0.01, w_c)
auto = ev[ev.valid_final & (ev.fin_seq >= tau)]
err = auto[~auto.exact_final]
print(f"しきい値 {tau:.3f}。評価データ {len(ev)}枚のうち、自動処理 {len(auto)}枚、そのうち誤り {len(err)}枚（{len(err) / len(auto):.2%}）")
print("誤りの難易度別:", err.difficulty.value_counts().to_dict())
cnt = Counter()
for g, p in zip(err["gt"], err["final"]):
    if len(g) == len(p):
        cnt.update((a, b) for a, b in zip(g, p) if a != b)
print("取り違え（正解→読み）:", [f"{a}→{b} ×{n}" for (a, b), n in cnt.most_common(10)])
print(f"誤りの確信度(seq)の平均 {err.fin_seq.mean():.2f} ／ 自動処理して正解だったものの平均 {auto[auto.exact_final].fin_seq.mean():.2f}")

**見るポイント**
- 自動処理の誤りは、**ほぼ「1文字だけの取り違え」**（6↔8、0↔6、5↔6、な↔ら、な↔は、ま↔さ…）。**形が似た文字**で、置き換えても**書式に合う**ため、書式検査では防げません。
- **確信度（seq）も、正解の自動処理（平均0.75）と、誤り（平均0.64）で、ほとんど変わりません。** 誤りは、**自信たっぷりに出てくる**ので、確信度でも防げません。
- → **AIの出力だけを見ていても、これ以上は減らせません。** 次は、**別の情報**を使います。

> 💡 **次の Phase5（入出庫の照合）が、ここに効きます。** 出庫する車は、**必ず入庫の記録がある**。読み間違えたナンバーは、**入庫記録に存在しない**（または、1文字違いの記録が1つある）ことが多い。**「入庫記録と一致するか」**という**システム側の情報**で、AIの誤りを見つけられるのです。AIの外側（データベース）が、AIを助けます。

## 🧪 試してみよう

### 演習1: 目標の誤り率を変える
**やること**: 「6. しきい値の決め方」の表は固定なので、次のセルを実行します。下の **`target = 0.01`** を **`0.005`**、**`0.02`**、**`0.05`** に書き換えて、そのたびに実行します。
- **見るポイント**: 目標を**厳しく**（小さく）すると、**自動処理率**はどうなりますか？ **要確認率**は？
- **分かること**: **誤りを減らすほど、人が確認する量が増える**（トレードオフ）。

In [ ]:
target = 0.01      # ← 0.005 / 0.02 / 0.05 に書き換えて実行
mix = "昼間中心(0.6:0.3:0.1)"
tau = choose_threshold(cal.fin_seq, cal.exact_final, cal.valid_final, target, weights(cal, mix))
r = apply(ev.fin_seq, ev.exact_final, ev.valid_final, tau, weights(ev, mix))
print(f"目標の誤り率 {target:.1%} → しきい値 {tau:.3f}")
pd.Series({k: round(v, 4) for k, v in r.items()}, name="評価データでの結果")

### 演習2: 誤課金の重さを変えてみる
**やること**: 「7. コストで決める」のセルの **`for N in [1, 5, 20, 100, 500]:`** を **`[2, 10, 50, 1000, 5000]`** に書き換えて実行します。
- **見るポイント**: N が非常に大きい（5000）とき、**自動処理率はどこまで下がりますか？** すべて人が確認する（自動処理率0）より、期待コストは小さいままですか？
- **分かること**: 損害が極端に大きいと、**ほとんど自動化できなくなる**。AIの精度向上の価値は、**損害の重さによって変わる**。

### 演習3: 3つの確信度を、曲線で比べる
**やること**: 「5. 振り分け」のグラフのセルの、**`"fin_seq"`** を **`"fin_min_max"`**、**`"fin_margin"`** に書き換えて実行し、曲線の位置を比べます（Cの行だけ書き換えれば十分です）。
- **見るポイント**: seq の曲線と比べて、どちらが上（悪い）ですか？ AUROC の表（0.87〜0.91）と合っていますか？
- **分かること**: 確信度の**作り方で、運用の効率が変わる**。

### 考えてみよう（コード不要）
1. **補正**の価値が小さかったのに、**検査**の価値が大きかったのは、なぜでしょう？ （ヒント: 「間違いに気付く」ことと、「間違いを直す」ことの難しさの違い）
2. しきい値を決めるための**較正セットを、実際のシステムで用意する**としたら、**何が大変**でしょう？ **少ないデータで決めるなら、どんな工夫**が必要でしょう？
3. Phase3で「**Hardは画像に情報が残っていない**」と分かりました。**読み取り不可・要確認**を、**撮影の側（カメラ・照明）の改善**と組み合わせるなら、どんな運用が考えられますか？

---

## ✅ まとめ

- **AIは間違える前提**で、**検査→確信度→振り分け**の仕組みを組み込んだ。
  - **書式検査**: 書式に合わないものは、**100%間違い**。**安全に弾ける**（正しい読み取りを弾かない）。
  - **補正**: 期待したほど**役に立たなかった**（元が壊れているときに、もっともらしい嘘を作る）。自動処理にしない。
  - **確信度**: **seq（文字列全体の確率）が、Hard と全体でわずかに優位**だが、4つに大差はなく、**完璧ではない**。
  - **振り分け**: 書式検査＋seq で、Phase3のままよりも、**同じ誤り率で、ずっと多く自動処理**できる。
- **しきい値は、評価とは別のデータで、十分な数で決める**。300枚では、**実際の誤り率が目標を大きく超えた**。
- **「誤り率の目標」は、損害の重さ（コスト）の裏返し。** **誤課金を重く見るほど、人に回す**。
- **残る誤りは、「形が似た文字の取り違え」**。**ルールでも確信度でも防げない**。→ **AIの外側の情報（入庫記録）**が必要。

➡ 次は **Phase5: 入出庫シミュレーション**。入庫を記録し、出庫時に**照合**し、料金を計算します。**入庫記録との照合で、読み間違いを見つけ**、誤課金率・未照合率・要確認率を、システム全体で測ります。